# 🚀 Real-ESRGAN GPU Upscaler Server

This notebook runs a FastAPI server with Real-ESRGAN 4x image upscaling.

**Steps:**
1. Enable GPU: Runtime → Change runtime type → GPU (T4)
2. Run all cells (Runtime → Run all)
3. Copy the public URL from the last cell
4. Use that URL in your backend

In [ ]:
# Install dependencies
!pip install -q fastapi uvicorn pyngrok pillow numpy opencv-python torch torchvision
!pip install -q realesrgan basicsr facexlib gfpgan
print("✅ Dependencies installed")

In [ ]:
# Download Real-ESRGAN model
!wget -q https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth -P /content/weights/
print("✅ Model downloaded")

In [ ]:
# Check GPU availability
import torch
if torch.cuda.is_available():
    print(f"✅ GPU Available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ No GPU detected. Enable GPU: Runtime → Change runtime type → GPU")

In [ ]:
# Create FastAPI server
%%writefile /content/upscaler_app.py
from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.responses import StreamingResponse, JSONResponse
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import torch
from PIL import Image
import io
import base64
import numpy as np
import cv2
from realesrgan import RealESRGANer
from basicsr.archs.rrdbnet_arch import RRDBNet

app = FastAPI(title="Real-ESRGAN GPU Upscaler")

# Enable CORS
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# Initialize Real-ESRGAN
model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=4)
upsampler = RealESRGANer(
    scale=4,
    model_path='/content/weights/RealESRGAN_x4plus.pth',
    model=model,
    tile=512,
    tile_pad=10,
    pre_pad=0,
    half=torch.cuda.is_available()
)

class Base64Image(BaseModel):
    image: str  # base64 encoded image

@app.get("/")
async def root():
    return {"message": "Real-ESRGAN GPU Upscaler", "status": "running", "gpu": torch.cuda.is_available()}

@app.get("/health")
async def health():
    return {
        "status": "healthy",
        "gpu_available": torch.cuda.is_available(),
        "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
    }

@app.post("/upscale")
async def upscale_image(file: UploadFile = File(...)):
    try:
        # Read image
        contents = await file.read()
        image = Image.open(io.BytesIO(contents)).convert('RGB')
        img_array = np.array(image)
        img_array = cv2.cvtColor(img_array, cv2.COLOR_RGB2BGR)
        
        # Upscale
        output, _ = upsampler.enhance(img_array, outscale=4)
        
        # Convert back to image
        output = cv2.cvtColor(output, cv2.COLOR_BGR2RGB)
        result_image = Image.fromarray(output)
        
        # Return as bytes
        img_byte_arr = io.BytesIO()
        result_image.save(img_byte_arr, format='PNG')
        img_byte_arr.seek(0)
        
        return StreamingResponse(img_byte_arr, media_type="image/png")
    
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/upscale-base64")
async def upscale_base64(data: Base64Image):
    try:
        # Decode base64
        image_data = base64.b64decode(data.image.split(',')[-1])
        image = Image.open(io.BytesIO(image_data)).convert('RGB')
        img_array = np.array(image)
        img_array = cv2.cvtColor(img_array, cv2.COLOR_RGB2BGR)
        
        # Upscale
        output, _ = upsampler.enhance(img_array, outscale=4)
        
        # Convert back to base64
        output = cv2.cvtColor(output, cv2.COLOR_BGR2RGB)
        result_image = Image.fromarray(output)
        img_byte_arr = io.BytesIO()
        result_image.save(img_byte_arr, format='PNG')
        img_byte_arr.seek(0)
        result_base64 = base64.b64encode(img_byte_arr.getvalue()).decode('utf-8')
        
        return JSONResponse({"upscaled_image": f"data:image/png;base64,{result_base64}"})
    
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8080)

In [ ]:
# Setup ngrok authentication (REQUIRED)
from pyngrok import ngrok, conf

# Get your free authtoken from: https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTH_TOKEN = ""  # Paste your token here

if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    print("✅ Ngrok authenticated")
else:
    print("⚠️ Please add your ngrok authtoken above")
    print("   Get it from: https://dashboard.ngrok.com/get-started/your-authtoken")

In [ ]:
# Start the server with ngrok tunnel
import threading
import uvicorn
import sys
sys.path.append('/content')
from upscaler_app import app

# Start FastAPI in background
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8080, log_level="info")

thread = threading.Thread(target=run_server, daemon=True)
thread.start()

# Wait for server to start
import time
time.sleep(3)

# Create ngrok tunnel
public_url = ngrok.connect(8080)

print("\n" + "="*60)
print("🎉 GPU UPSCALER SERVER IS RUNNING!")
print("="*60)
print(f"\n📡 Public URL: {public_url}")
print(f"\n🔗 Endpoints:")
print(f"   Health Check: {public_url}/health")
print(f"   Upscale (file): {public_url}/upscale")
print(f"   Upscale (base64): {public_url}/upscale-base64")
print(f"\n📋 Add this URL to your backend server.py:")
print(f"   GPU_UPSCALER_URL = \"{public_url}\"")
print("\n⚠️ Keep this notebook running! Server stops when notebook stops.")
print("="*60)

# Keep the cell running
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\n👋 Server stopped")

## 🧪 Test the Server

Run this cell to test if the upscaler is working:

In [ ]:
# Test the upscaler
import requests

# Replace with your public URL from above
SERVER_URL = ""  # e.g., "https://abc123.ngrok-free.app"

if SERVER_URL:
    response = requests.get(f"{SERVER_URL}/health")
    print("Health Check:", response.json())
else:
    print("⚠️ Please add the server URL above")